# SceneHawk - fill the feelings table

Runs the trained feelings model over every review in Supabase, averages the
scores per film, and writes `film_feelings`. This is step 12 of the pipeline
(`scripts/12_score_reviews.py`); the notebook only sets it up on a GPU.

**Before you start**

1. Runtime > Change runtime type > **T4 GPU**.
2. The model from `train_feelings.ipynb` is in Google Drive under
   `MyDrive/scenehawk/models/<version>/`. `MODEL_NAME` in the next cell says
   which one; it is set to the retrained model picked on 2026-11-08.
   Folder names do not sort by date, so the notebook never guesses.
3. Add your Supabase connection string as a Colab secret: the key icon in the
   left bar > **Add new secret**, name `SUPABASE_DB_URL`, value from Supabase >
   Connect > Connection pooling (with your password in it). Turn on notebook
   access. Never paste it into a cell.
4. `db/schema.sql` and `db/migrations/2026-11-06_search.sql` have been run in
   the Supabase SQL editor.

The code is cloned from `REPO` on GitHub, so changes to
`scripts/12_score_reviews.py` only take effect once they are pushed to
`BRANCH`. The scoring step prints how reviews are cut: for the retrained model
it must say `head_tail`.

Then **Runtime > Run all**. About 15 minutes for ~37k reviews. If Colab
disconnects, run all again: finished reviews are cached in Drive and skipped.

In [ ]:
# Where the code comes from. Change these if the team repo moves.
REPO = "https://github.com/notLogi/SceneHawk-CS499-Project.git"
BRANCH = "main"

from google.colab import drive
drive.mount("/content/drive")

import pathlib
# The model to fill the table with. Set by name: folder names do not sort by
# date (distilroberta_e16_... sorts after distilroberta_e12_...).
MODEL_NAME = "distilroberta_e12_headtail_s49900-20261108-2130"

ROOT = pathlib.Path("/content/drive/MyDrive/scenehawk/models")
print("saved models:")
for m in sorted(ROOT.glob("*/config.json")):
    print(" ", m.parent.name)
MODEL_DIR = str(ROOT / MODEL_NAME)
assert (ROOT / MODEL_NAME / "config.json").exists(), f"{MODEL_NAME} not found in {ROOT}"
print("using:", MODEL_DIR)

In [ ]:
!rm -rf /content/repo && git clone -q --depth 1 -b {BRANCH} {REPO} /content/repo
!pip -q install psycopg2-binary python-dotenv

import os
from google.colab import userdata
os.environ["SUPABASE_DB_URL"] = userdata.get("SUPABASE_DB_URL")

# Keep the per-review cache in Drive so a disconnect doesn't lose work.
cache = "/content/drive/MyDrive/scenehawk/feelings"
os.makedirs(cache, exist_ok=True)
!rm -rf /content/repo/chatbot/data/feelings && ln -s {cache} /content/repo/chatbot/data/feelings

## Score the reviews and write the table

The first lines print the model version and the device. It should say `cuda`.
At the end it prints the top films for a few feelings as a sanity check.

In [ ]:
%cd /content/repo/chatbot
!python scripts/12_score_reviews.py --model "{MODEL_DIR}" --batch 64

## Then

- The website's chat page now answers from the new scores. Check it at
  `<your site>/api/chat` (it reports how many films have feeling scores).
- Retrained the model? Run this notebook again. Each model version writes its
  own cache file, and the table is replaced film by film.